**E-commerce RAG Notebook (Step-by-Step)**

**1. Install dependencies (run once) :**
**uv add langchain langchain-openai langchain-chroma pandas python-dotenv**

**2. Load environment variables**

In [35]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

**📦 3. Load dataset**

In [36]:
import pandas as pd

df = pd.read_csv("data/product.csv")
df.head()

,name,price,brand,category,description
0,iPhone 13,52000,Apple,Smartphone,Premium iPhone with excellent camera performan...
1,iPhone 12,45000,Apple,Smartphone,Apple smartphone with great camera and solid p...
2,Samsung Galaxy S21,30000,Samsung,Smartphone,Flagship phone with AMOLED display and powerfu...
3,Samsung Galaxy M34,18000,Samsung,Smartphone,Mid-range phone with huge battery and good per...
4,Redmi Note 12,15000,Xiaomi,Smartphone,Budget phone with strong battery life and dece...


**🧠 4. Convert to LangChain Documents**

In [37]:
from langchain_core.documents import Document

docs = []

for _, row in df.iterrows():
    text = f"""
    Product: {row['name']}
    Brand: {row['brand']}
    Category: {row['category']}
    Price: {row['price']} INR
    Description: {row['description']}
    """
    
    docs.append(
        Document(
            page_content=text,
            metadata={
                "name": row["name"],
                "price": row["price"],
                "category": row["category"],
                "brand": row["brand"]
            }
        )
    )

len(docs)

33

**⚙️ 5. Create Embeddings + Vector DB**

In [38]:
from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="db"
)

print("Vector DB created")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7357.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB created


**🔍 6. Test Retrieval (VERY IMPORTANT STEP)**

In [39]:
retriever = db.as_retriever(search_kwargs={"k": 3})

query = "best phone under 20000"

results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print("------")


    Product: Asus ROG Phone 6
    Brand: Asus
    Category: Smartphone
    Price: 60000 INR
    Description: Ultimate gaming phone with high refresh rate and cooling system
    
------

    Product: Asus ROG Phone 6
    Brand: Asus
    Category: Smartphone
    Price: 60000 INR
    Description: Ultimate gaming phone with high refresh rate and cooling system
    
------

    Product: Asus ROG Phone 6
    Brand: Asus
    Category: Smartphone
    Price: 60000 INR
    Description: Ultimate gaming phone with high refresh rate and cooling system
    
------


**🤖 7. Add LLM**

In [40]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

**🧠 8. Build RAG Function**

In [41]:
def ask(query):
    try:
        # Step 1: Limit retrieved docs
        docs = retriever.invoke(query)
        docs = docs[:3]  # ✅ LIMIT (very important)

        # Step 2: Clean & trim content
        context_parts = []
        for d in docs:
            text = d.page_content.strip().replace("\n", " ")
            context_parts.append(text[:300])  # ✅ limit each doc

        context = "\n\n".join(context_parts)

        # Step 3: Better prompt (short & structured)
        prompt = f"""You are an intelligent e-commerce assistant.

Rules:
- Recommend 2-3 best products
- Stay within budget if mentioned
- Keep answer short and clear

Context:
{context}

User Query:
{query}
"""

        # Step 4: Safe invoke (prevents crash)
        response = llm.invoke(prompt)

        return response.content

    except Exception as e:
        print("Error:", e)
        return "Sorry, something went wrong. Please try again."
    
ask("best phone under 20000")

'None of the listed options fit your 20,000 INR budget—the Asus ROG Phone 6 is priced at 60,000 INR. Would you like me to find the top smartphones under 20,000 INR from our catalog?'

In [42]:
ask("best phone under 20000")

'No options under ₹20,000 in the current listing. The available phones are ₹60,000. Want me to search for smartphones under ₹20k or show closer options under ₹25k/₹30k?'

In [45]:
ask("best phone under 100000")

'Best option under 100000 from your catalog: Samsung Galaxy M34 — 18000 INR. A solid mid-range with a huge battery and good performance.\n\nIf you’d like more choices, I can broaden the search under 100000.'

In [46]:
ask("best phone under 15000 to 20000")

'Best pick under 15k-20k:\n- Samsung Galaxy M34 — 18,000 INR. Mid-range with a huge battery and good performance.\n\nNote: This is the only option in the current list within this price range. Want me to search more brands/models under 20k?'

In [ ]:
ask("best phone")

**🧪 9. Test your chatbot**

**🔥 10. Upgrade → JSON Output (Important)**

In [43]:
def ask_json(query):
    docs = retriever.invoke(query)
    
    context = "\n\n".join([d.page_content for d in docs])
    
    prompt = f"""
Return answer in JSON format:

[
  {{
    "name": "...",
    "price": "...",
    "reason": "..."
  }}
]

Context:
{context}

Query:
{query}
"""
    
    return llm.invoke(prompt).content

In [44]:
ask_json("best phone under 20000 with good camera")

'[\n  {\n    "name": "Samsung Galaxy M14 5G",\n    "price": "Under 20,000 INR (price varies by offers)",\n    "reason": "Best balance of camera quality, software stability, and value under 20k; reliable daylight photos and capable night mode, plus solid battery life."\n  },\n  {\n    "name": "Redmi Note 12 Pro 5G",\n    "price": "Under 20,000 INR (on sale/offer)",\n    "reason": "Versatile multi-camera setup for the price with strong daylight shots and good overall image processing; great value in this segment."\n  },\n  {\n    "name": "Poco X5 5G",\n    "price": "Under 20,000 INR (on sale/offer)",\n    "reason": "Competitive camera performance for the price, solid overall performance and good value in budget range."\n  }\n]'